In [ ]:
# Dependencies: doubleml, lightning, linearmodels (install locally)
print('Local mode')

In [ ]:
# Local mode - no Google Drive needed
print('Local mode')

In [ ]:
import os, sys
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Working directory: {os.getcwd()}')

# Notebook 04v2 — DoubleML Evaluation: Price Elasticity of Demand
# Revision: Delta Variables + Neighbor/Competitor Prices

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**
Applied to: Amazon Women's Shoes (Size 8)

---

## What Changed in This Version

**Revision 1 — Delta Variables (previous fix):**
Switched outcome/treatment from levels (Q_t, P_bb_t) to first differences
(Delta_Q_t, Delta_P_bb_t) to remove the 0.933 lag suppression problem.
Result: ATE moved from -0.066 to -0.094.

**Revision 2 — Neighbor/Competitor Prices (this version):**
Added neighbor_price_t and Delta_neighbor_price_t as controls.
For each ASIN, we compute the mean price of its top-10 most similar
products (by embedding cosine similarity) at each time period.

Why this helps (Bach et al. 2025, Appendix C):
- Competitor price changes are correlated with our price (shared market)
- But do not directly cause our product's demand (exogenous variation)
- Gives DoubleML IV-like price variation to identify theta more accurately

Expected improvement: ATE closer to -0.2 to -0.5

---

## Model Specification

```
Delta_Q_it = θ · Delta_P_it + g(X_it) + ε_it
```
- Outcome Y = Delta_Q_t
- Treatment D = Delta_P_bb_t
- Controls X = embeddings + tabular + neighbor_price_t + Delta_neighbor_price_t
- window=28 kept unchanged


In [ ]:
import os
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity  # ← ADDED for neighbor prices

import doubleml as dml

from utils.utils_data2 import (
    get_cis,
    generate_basis,
    generate_interactions,
)

from utils.utils_models import (
    iv_model,
    lin_model,
    lin_model_cate,
    summarize_lin_model,
    summarize_dml,
    chi2_test,
    predict_cate,
)
import warnings
warnings.filterwarnings("ignore")

palette = sns.color_palette("colorblind")

confidence_level = 0.9
ci_level_name, ci_names = get_cis(confidence_level)

output_dir = "../output/04_evaluation_v2_delta"
Path(output_dir).mkdir(parents=True, exist_ok=True)


In [ ]:
txt_only = False
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{include_embeddings}"
df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2",
    "P_bb_t-2",
    "REVIEW_COUNT_t-2",
    "RATING_t-2",
    "pred_ml_l_lag_1",
    "pred_ml_m_lag_1",
]


df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val = df_full_val.dropna()

dummy_subcat_names = [
    category for category in df_full_val["subcat_aggregated"].unique()
]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

In [ ]:
print(dataframe_name)

In [ ]:
print(df_full_val.shape)
print(df_full_train.shape)

In [ ]:
df_full_val[["ASIN", "date"]].to_csv("main_val_keys.csv", index=False)
df_full_train[["ASIN", "date"]].to_csv("main_train_keys.csv", index=False)

In [ ]:
df_full_val.isna().sum()

In [ ]:
df_full_train["date"].unique()

In [ ]:
df_full_val["date"].unique()

In [ ]:
df_full_train["ASIN"].nunique()

In [ ]:
df_full_val["ASIN"].nunique()

In [ ]:
from datetime import datetime

date_objects = [datetime.strptime(date, "%Y-%m-%d") for date in all_time_steps]

# Calculate the difference in days between consecutive dates
date_differences = [
    (date_objects[i + 1] - date_objects[i]).days for i in range(len(date_objects) - 1)
]

# Print the differences
print(date_differences)

## Variable Definitions (Revised — Delta + Neighbor Prices)

| Variable | Description | Role in model |
|---|---|---|
| `Delta_Q_t` | Q_t − Q_t-1 — change in log sales rank | **Outcome Y** |
| `Delta_P_bb_t` | P_bb_t − P_bb_t-1 — change in log buybox price | **Treatment D** |
| `neighbor_price_t` | Mean price of top-k similar ASINs at time t | **New control** |
| `Delta_neighbor_price_t` | Change in neighbor price vs last period | **New control** |
| `emb_0...255` | 256-dim multimodal embedding (txt+img+tab) | Controls X |
| `similarity_cluster_0...4` | Cosine similarity to 5 KMeans centroids | CATE basis |
| `RATING_t-1`, `REVIEW_COUNT_t-1` | Lagged product quality signals | Controls X |
| `dummy_subcat_*` | Subcategory fixed effects | Controls X |
| `dummy_time_*` | Time fixed effects (28-day periods) | Controls X |

**Why neighbor prices help:**
Bach et al. (2025) Appendix C shows that adding substitute/competitor prices
strengthens DoubleML identification. A neighbor's price change is correlated
with our product's price (shared market conditions) but does not directly
cause our product's demand — making it an IV-like control that adds
exogenous price variation for DoubleML to work with.

**Removed from controls:**
- ~~`Q_t-1`~~ — was causing 0.933 lag suppression
- ~~`P_bb_t-1`~~ — already embedded in Delta_P_bb_t

**window=28 kept unchanged.**


In [ ]:
n_lags = 1

# ── REVISION v2: Delta variables + Neighbor Prices ────────────────────────
# outcome = Delta_Q_t, treatment = Delta_P_bb_t (from previous revision)
# NEW: compute neighbor/competitor prices from embedding similarity
# REFERENCE: Bach et al. (2025) Appendix C — substitute prices improve ATE
# ─────────────────────────────────────────────────────────────────────────

outcome        = ["Delta_Q_t"]     # REVISED: first difference of quantity
base_treatment = ["Delta_P_bb_t"]  # REVISED: first difference of price

# lag_1_vars kept for reference only — NOT used in PLR controls
lag_1_vars = ["Q_t-1", "P_bb_t-1"]

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + [
        "Lightning Deals: Upcoming Deal",
        "Buy Box: Is FBA",
    ]
)
dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

embeddings = [f"emb_{i}" for i in range(256)]
controls_to_scale = cont_controls + add_controls_to_scale
additional_controls = cont_controls + dummy_controls

# ── NEIGHBOR PRICE COMPUTATION ─────────────────────────────────────────────
# Step 1: Get unique ASINs and their embeddings (one row per ASIN)
# Use val set only — consistent with DoubleML evaluation set

print("Computing neighbor prices from embedding similarity...")

# Get one embedding vector per ASIN (time-independent)
asin_emb = (
    df_full_val[["ASIN"] + embeddings]
    .groupby("ASIN")[embeddings]
    .mean()  # average across time periods (embeddings are stable)
    .reset_index()
)

asin_list = asin_emb["ASIN"].values
emb_matrix = asin_emb[embeddings].values

# Step 2: Compute cosine similarity between all ASINs
# Shape: (n_asins, n_asins) — e.g. 2574 × 2574
print(f"  Computing {len(asin_list)} x {len(asin_list)} similarity matrix...")
sim_matrix = cosine_similarity(emb_matrix)  # values in [-1, 1]

# Set diagonal to -1 so each ASIN doesn't count as its own neighbor
np.fill_diagonal(sim_matrix, -1)

# Step 3: For each ASIN, find top-k nearest neighbors (k=10)
K = 10
print(f"  Finding top-{K} neighbors per ASIN...")
top_k_indices = np.argsort(sim_matrix, axis=1)[:, -K:]  # shape: (n_asins, K)

# Step 4: For each ASIN at each time period, compute mean neighbor price
# Build ASIN → neighbor ASIN list mapping
asin_to_idx = {asin: i for i, asin in enumerate(asin_list)}

# Get price per ASIN per date from val set
price_pivot = (
    df_full_val[["ASIN", "date", "P_bb_t"]]
    .drop_duplicates(subset=["ASIN", "date"])
    .pivot(index="ASIN", columns="date", values="P_bb_t")
)

print(f"  Computing neighbor mean prices across {len(price_pivot.columns)} time periods...")

neighbor_price_rows = []
for asin in asin_list:
    if asin not in asin_to_idx:
        continue
    idx = asin_to_idx[asin]
    neighbor_indices = top_k_indices[idx]
    neighbor_asins = asin_list[neighbor_indices]
    # Get valid neighbors that exist in price_pivot
    valid_neighbors = [a for a in neighbor_asins if a in price_pivot.index]
    if len(valid_neighbors) == 0:
        continue
    neighbor_prices = price_pivot.loc[valid_neighbors].mean(axis=0)  # mean across neighbors
    for date, price in neighbor_prices.items():
        neighbor_price_rows.append({
            "ASIN": asin,
            "date": date,
            "neighbor_price_t": price
        })

df_neighbor_price = pd.DataFrame(neighbor_price_rows)

# Step 5: Compute Delta_neighbor_price_t (change in neighbor price)
df_neighbor_price = df_neighbor_price.sort_values(["ASIN", "date"]).reset_index(drop=True)
df_neighbor_price["Delta_neighbor_price_t"] = (
    df_neighbor_price.groupby("ASIN")["neighbor_price_t"].diff()
)

# Step 6: Merge back into df_full_val and df_full_train
df_full_val = df_full_val.merge(
    df_neighbor_price[["ASIN", "date", "neighbor_price_t", "Delta_neighbor_price_t"]],
    on=["ASIN", "date"],
    how="left"
)

# For train set: use same neighbor mapping (neighbors computed from val embeddings)
# This is acceptable since embeddings are product-level and time-independent
df_full_train = df_full_train.merge(
    df_neighbor_price[["ASIN", "date", "neighbor_price_t", "Delta_neighbor_price_t"]],
    on=["ASIN", "date"],
    how="left"
)

# Drop rows where neighbor price is missing (first time period has no delta)
df_full_val   = df_full_val.dropna(subset=["Delta_neighbor_price_t"]).reset_index(drop=True)
df_full_train = df_full_train.dropna(subset=["Delta_neighbor_price_t"]).reset_index(drop=True)

# Add neighbor price to continuous controls
neighbor_controls = ["neighbor_price_t", "Delta_neighbor_price_t"]
additional_controls = cont_controls + dummy_controls + neighbor_controls  # UPDATED

print(f"  Done! Val shape after adding neighbor prices: {df_full_val.shape}")
print(f"  Train shape: {df_full_train.shape}")
print(f"  Sample neighbor prices (val):")
print(df_full_val[["ASIN", "date", "neighbor_price_t", "Delta_neighbor_price_t"]].head(5).to_string())


## Interactions and CATE Basis Variables (Revised)

CATE basis uses **cluster similarities only** — lag variables removed
since they are no longer meaningful after switching to Delta outcome.

**Basis variables (`df_basis`):**
- `similarity_cluster_0...4` — cosine similarity to each of 5 KMeans centroids

**Removed from basis:**
- ~~`Q_t-1`~~ — lagged demand level (not meaningful in a diff model)
- ~~`P_bb_t-1`~~ — lagged price level (already differenced out)


In [ ]:
# ── REVISION: lag_1_vars removed from basis ─────────────────────────────
# ORIGINAL: basis_vars = controls_similarities + lag_1_vars
# REVISED:  basis_vars = controls_similarities only

basis_vars = controls_similarities  # REVISED

df_basis, column_transformer, column_dict = generate_basis(
    df=df_full_val,
    cols_without_scaling=controls_similarities,
    cols_to_scale=[],   # REVISED: no lag vars
    degree=1,
)

# ── Index alignment fix ───────────────────────────────────────────────────
# Reset index after basis generation to ensure alignment with df_dynamic
df_basis = df_basis.reset_index(drop=True)

df_basis.head()


In [ ]:
# ── REVISION: No lag-based interactions ─────────────────────────────────
# ORIGINAL: cols_to_interact=["Q_t-1", "P_bb_t-1"]
# REVISED:  cols_to_interact=[] — similarities only, no lag interactions

df_interact_val, interaction_vars = generate_interactions(
    df=df_full_val, df_basis=df_basis, cols_to_interact=[]  # REVISED
)

print(f"Interactions: {interaction_vars}")
df_interact_val.columns


## Average Treatment Effect (ATE) Estimation

We estimate the **average price elasticity** across all products using
two families of models:

1. **Linear models** — OLS with various control specifications
   (no lag, with lag, with embeddings, with controls)
2. **DoubleML PLR** — Partially Linear Regression using LightGBM
   as the nuisance model learner

The linear models serve as a sanity check and a baseline.
The DoubleML models are the main causal estimates following the paper.

**Confidence level:** 90% throughout (following Bach et al. 2025)

In [ ]:
df_dynamic = df_interact_val.copy()

# ── Index alignment fix ───────────────────────────────────────────────────
# Ensure df_dynamic and df_basis share the same clean integer index
# Required by lin_model_cate (statsmodels is strict about index alignment)
df_dynamic = df_dynamic.reset_index(drop=True)
df_basis    = df_basis.reset_index(drop=True)


### Linear Model Baselines

We run 6 OLS specifications with increasing control richness.
Each prints the price coefficient on `P_bb_t` — the raw elasticity estimate.

**What to expect:**
- No controls: most negative (omitted variable bias inflates elasticity)
- Adding embeddings: coefficient shrinks (embeddings absorb product heterogeneity)
- Adding lag variables: R² jumps to ~83% (past rank predicts current rank strongly)
- Adding both: most precise estimate with best R²

Note: Linear OLS is biased due to price endogeneity.
The DoubleML models below correct for this.

In [ ]:
def estimate_linear_model(controls):
    model = lin_model(
        df=df_dynamic,
        outcome=outcome,
        treatments=base_treatment,
        controls=controls,
        level=confidence_level,
        cov_type="cluster",
        cov_kwds={"groups": df_dynamic["ASIN"]},
    )
    return model

In [ ]:
estimate_linear_model(controls=[])

In [ ]:
estimate_linear_model(controls=embeddings)

In [ ]:
estimate_linear_model(controls=additional_controls)

In [ ]:
estimate_linear_model(controls=additional_controls + embeddings)

In [ ]:
ci_linear_base_lagged = estimate_linear_model(controls=lag_1_vars)

In [ ]:
ci_linear_embeddings = estimate_linear_model(controls=lag_1_vars + embeddings)

In [ ]:
ci_linear_base_lagged_add_controls = estimate_linear_model(
    controls=lag_1_vars + additional_controls
)

In [ ]:
ci_linear_base_lagged_add_controls_emb = estimate_linear_model(
    controls=lag_1_vars + additional_controls + embeddings
)

In [ ]:
ci_linear_base_lagged_add_controls_sim = estimate_linear_model(
    controls=lag_1_vars + controls_similarities + additional_controls
)

In [ ]:
ci_linear_interactions = estimate_linear_model(
    controls=interaction_vars + controls_similarities + additional_controls
)

In [ ]:
ci_linear_base_lagged["model"] = "Linear lagged"
ci_linear_embeddings["model"] = "Linear lagged_and_embeddings"
ci_linear_base_lagged_add_controls["model"] = "Linear lagged_and_add_controls"
ci_linear_base_lagged_add_controls_emb["model"] = (
    "Linear lagged_and_add_controls_and_emb"
)
ci_linear_base_lagged_add_controls_sim["model"] = (
    "Linear lagged_and_add_controls_and_sim"
)
ci_linear_interactions["model"] = "Linear interactions"

df_base = pd.concat(
    [
        ci_linear_base_lagged,
        ci_linear_embeddings,
        ci_linear_base_lagged_add_controls,
        ci_linear_base_lagged_add_controls_emb,
        ci_linear_base_lagged_add_controls_sim,
    ],
    ignore_index=True,
)

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
# Plot Estimate with CI
plt.errorbar(
    df_base["model"],
    df_base["coef"],
    yerr=[df_base["coef"] - df_base["lower"], df_base["upper"] - df_base["coef"]],
    fmt="o",
    capsize=5,
    capthick=2,
    ecolor=palette[1],
    color=palette[0],
    label=f"Estimate with {round(confidence_level * 100, 1)}% CI",
    zorder=2,
)

plt.title(f"Estimated ATE, and {round(confidence_level * 100, 1)}% Confidence Interval")
plt.xlabel("Treatment Level")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, f"linear_ate_estimates.png"))
plt.show()

In [ ]:
df_base

### DoubleML Partially Linear Regression (PLR) — Main Results

**Specification: Delta_Q_t ~ θ · Delta_P_bb_t + g(X) + ε**

> Outcome = Delta_Q_t | Treatment = Delta_P_bb_t
> lag_1_vars removed | neighbor_price_t and Delta_neighbor_price_t added to controls

**Why neighbor prices improve identification:**
Competitor price changes are correlated with our product's price (shared market)
but do not directly cause our product's demand — providing IV-like exogenous
variation that DoubleML can exploit for better theta identification.
(Reference: Bach et al. 2025 Appendix C)

**5 specifications tested:**

| Spec | Controls | ATE | Significant? |
|---|---|---|---|
| PLR + Embeddings | emb_* | TBD | TBD |
| PLR + Controls | tabular + neighbor | TBD | TBD |
| PLR + Emb + Controls | emb + tabular + neighbor | **TBD** | TBD |
| PLR + Sim + Controls | sim + tabular + neighbor | TBD | TBD |
| PLR + PCA + Controls | pca + tabular + neighbor | TBD | TBD |

Expected: closer to -0.2 to -0.5 with neighbor prices added


In [ ]:
ml_g = LGBMRegressor(n_estimators=500, lr=0.02, random_state=42, verbose=-1)
ml_m = LGBMRegressor(n_estimators=500, lr=0.02, random_state=42, verbose=-1)


def r2(y_true, y_pred):
    subset = np.logical_not(np.isnan(y_true))
    return r2_score(y_true[subset], y_pred[subset])


def estimate_dml_model(controls):
    tab_dml_data = dml.DoubleMLClusterData(
        df_dynamic,
        y_col=outcome[0],
        d_cols=base_treatment,
        x_cols=controls,
        cluster_cols="ASIN",
    )

    np.random.seed(3141)
    tab_dml_obj = dml.DoubleMLPLR(tab_dml_data, ml_g, ml_m)
    tab_dml_obj.fit()

    tab_summary_df = summarize_dml(tab_dml_obj, level=confidence_level)
    ci_tab_dml = tab_summary_df[ci_names + ["coef"]]
    ci_tab_dml.columns = ["lower", "upper", "coef"]

    r2_scores = tab_dml_obj.evaluate_learners(metric=r2)
    print(
        f"R2 Outcome: {round(r2_scores['ml_l'][0, 0], 4)} R2 Treatment: {round(r2_scores['ml_m'][0, 0], 4)}\n"
    )

    return tab_dml_obj, ci_tab_dml

In [ ]:
# REVISED: removed lag_1_vars (was: controls=lag_1_vars + embeddings)
tab_dml_obj_embeddings, ci_tab_dml_embeddings = estimate_dml_model(
    controls=embeddings
)


**PLR + Embeddings only** (lag1 + emb_256): ATE = **-0.088** (significant)
Nuisance R²: Q = 91.0%, P = 88.5%
Embeddings alone (without tabular controls) give a strong signal.

In [ ]:
# REVISED: removed lag_1_vars (was: controls=lag_1_vars + additional_controls)
tab_dml_obj_add_controls, ci_tab_dml_add_controls = estimate_dml_model(
    controls=additional_controls
)


In [ ]:
# REVISED: removed lag_1_vars (was: controls=lag_1_vars + embeddings + additional_controls)
tab_dml_obj_embeddings_add_controls, ci_tab_dml_embeddings_add_controls = (
    estimate_dml_model(controls=embeddings + additional_controls)
)


**PLR + Controls only** (lag1 + tabular): ATE = **-0.075** (significant)
Nuisance R²: Q = 93.4%, P = 89.8%
Tabular controls alone already identify a significant negative elasticity.

In [ ]:
# REVISED: removed lag_1_vars (was: controls=lag_1_vars + controls_similarities + additional_controls)
tab_dml_obj_sim_add_controls, ci_tab_dml_sim_add_controls = estimate_dml_model(
    controls=controls_similarities + additional_controls
)


In [ ]:
# REVISED: removed lag_1_vars (was: controls=lag_1_vars + controls_pca + additional_controls)
tab_dml_obj_pca_add_controls, ci_tab_dml_pca_add_controls = estimate_dml_model(
    controls=controls_pca + additional_controls
)


**PLR + Embeddings + Controls** (emb + tabular) — BEST MODEL (Revised)
Update ATE value here after running the cell above.
Expected: closer to -0.3 to -0.8 vs original -0.066


### ATE Visualization — Coefficient Plot

The plot shows all linear and DoubleML estimates together with 90% CI.

**Reading the plot:**
- Points further from zero = stronger elasticity
- Wider bars = more uncertain estimates
- Models where the CI crosses zero = not statistically significant

**Key observation:** Adding embeddings consistently makes the price
coefficient more negative — embeddings absorb product quality variation
that would otherwise confound the price-demand relationship.

In [ ]:
ci_tab_dml_embeddings["model"] = "plr_embeddings"
ci_tab_dml_add_controls["model"] = "plr_add_controls"
ci_tab_dml_embeddings_add_controls["model"] = "plr_embeddings_add_controls"
ci_tab_dml_sim_add_controls["model"] = "plr_sim_add_controls"
ci_tab_dml_pca_add_controls["model"] = "plr_pca_add_controls"


df_averages = pd.concat(
    [
        ci_tab_dml_embeddings,
        ci_tab_dml_add_controls,
        ci_tab_dml_embeddings_add_controls,
        ci_tab_dml_sim_add_controls,
        ci_tab_dml_pca_add_controls,
    ],
    ignore_index=True,
)

**PLR + Similarities + Controls** (lag1 + sim + tabular): ATE = **-0.079** (significant)
Nuisance R²: Q = 93.7%, P = 90.2%
Using compressed cluster similarities gives a slightly stronger estimate than full embeddings.

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
# Plot Estimate with CI
plt.errorbar(
    df_averages["model"],
    df_averages["coef"],
    yerr=[
        df_averages["coef"] - df_averages["lower"],
        df_averages["upper"] - df_averages["coef"],
    ],
    fmt="o",
    capsize=5,
    capthick=2,
    ecolor=palette[1],
    color=palette[0],
    label=f"Estimate with {round(confidence_level * 100, 1)}% CI",
    zorder=2,
)

plt.title(f"Estimated ATE, and {round(confidence_level * 100, 1)}% Confidence Interval")
plt.xlabel("Treatment Level")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, f"plr_ate_estimates.png"))
plt.show()

## GATES — Group Average Treatment Effects

GATES answers: *"Does price sensitivity differ by product cluster?"*

**Method:** Uses `tab_dml_obj_embeddings` (PLR + Embeddings spec)
and calls `.gate(groups)` where each product is assigned to its
closest cluster via `idxmax()` on `similarity_cluster_*` columns.

**Cluster source:** Multimodal embeddings (txt + img + tabular),
time-independent, 256 dimensions, KMeans k=5.

**What to look for:**
- Clusters with more negative GATE = more price elastic
- If CI excludes zero → that cluster's elasticity is statistically significant
- Large spread across clusters → strong heterogeneity in price sensitivity

**Our finding:** Budget shoe clusters (Clusters 0, 2 at ~$21-33)
are more elastic than premium clusters (Clusters 1, 3 at ~$40-45).

**PLR + PCA + Controls** (lag1 + pca + tabular): ATE = **-0.060** (significant)
Nuisance R²: Q = 93.9%, P = 89.4%
PCA-reduced embeddings (5 components) still give a significant estimate, but slightly weaker
than the full 256-dim embedding.

In [ ]:
groups = pd.DataFrame(
    df_dynamic[
        [
            "similarity_cluster_0",
            "similarity_cluster_1",
            "similarity_cluster_2",
            "similarity_cluster_3",
            "similarity_cluster_4",
        ]
    ].idxmax(axis=1)
)

print(groups.groupby(0).size())
gates_0 = tab_dml_obj_embeddings.gate(groups)
gates_0_summ = gates_0.summary
gates_0_summ["err_lower"] = gates_0_summ["coef"] - gates_0_summ["[0.025"]
gates_0_summ["err_upper"] = gates_0_summ["0.975]"] - gates_0_summ["coef"]

# Positions on the x-axis
x_vals = range(len(gates_0_summ))

plt.figure(figsize=(8, 5))
plt.errorbar(
    x=x_vals,
    y=gates_0_summ["coef"],
    yerr=[gates_0_summ["err_lower"], gates_0_summ["err_upper"]],
    fmt="o",
    capsize=5,
    color="blue",
    ecolor="black",
)

# Draw a zero line for reference
plt.axhline(y=0, color="gray", linestyle="--")

# Customizing x-axis labels
plt.xticks(
    ticks=x_vals,
    labels=[gr.replace("similarity", "") for gr in gates_0_summ.index],
    rotation=30,
    ha="right",
)
plt.ylabel("Coefficient Estimate")
plt.tight_layout()
plt.show()

## CATEs — Conditional Average Treatment Effects

CATEs go one level deeper — estimating an **individual elasticity
for every product observation** in the val set.

**Three models estimated:**

1. **Linear Specification** — linear BLP on similarity + lag basis
2. **Linear Specification (Interactions)** — adds interaction terms
3. **PLR Specification** — BLP on top of DoubleML residuals
   (uses `plr_obj = tab_dml_obj_embeddings_add_controls`)

**Key results from our data:**
- Mean CATE: -0.142 (std: 0.037)
- Range: [-0.397, +0.082]
- Positive CATE values → some products show perverse elasticity
  (possibly niche or premium products with inelastic demand)

**The CATE distribution tells us:** There is variation in price sensitivity
across individual products — the average elasticity of -0.066 masks
variation across shoe types and price tiers.

In [ ]:
uniform_ci = True  # False | True

In [ ]:
# REVISED: lag_1_vars removed, neighbor_controls included
# neighbor_price_t and Delta_neighbor_price_t are now in additional_controls
# so cate_controls automatically includes them

cate_controls = embeddings + additional_controls              # includes neighbor prices
cate_controls_interactions = cate_controls + interaction_vars

plr_obj = tab_dml_obj_embeddings_add_controls


In [ ]:
import gc

gc.collect()

In [ ]:
cate_kwargs_val = {"cov_type": "cluster", "cov_kwds": {"groups": df_dynamic["ASIN"]}}

cate_models_polynomial = {
    "Linear Specification": None,
    "Linear Specification (Interactions)": None,
    "PLR Specification": None,
}

# Linear Models
cate_models_polynomial["Linear Specification"] = lin_model_cate(
    df=df_dynamic,
    outcome=outcome,
    basis=df_basis,
    treatment=base_treatment,
    controls=cate_controls,
    **cate_kwargs_val,
)

print(f"\nLinear Specification")
df_specification_1 = summarize_lin_model(
    cate_models_polynomial["Linear Specification"], level=confidence_level
)
print("=" * 80)  #

cate_models_polynomial["Linear Specification (Interactions)"] = lin_model_cate(
    df=df_dynamic,
    outcome=outcome,
    basis=df_basis,
    treatment=base_treatment,
    controls=cate_controls_interactions,
    **cate_kwargs_val,
)

print(f"\nLinear specification with interactions")
df_specification_2 = summarize_lin_model(
    cate_models_polynomial["Linear Specification (Interactions)"],
    level=confidence_level,
)
print("=" * 80)

# tabular DML model
cate_models_polynomial["PLR Specification"] = plr_obj.cate(df_basis, **cate_kwargs_val)

print(f"\nTabular DML PLR Specification")
df_specification_3 = summarize_lin_model(
    cate_models_polynomial["PLR Specification"].blp_model, level=confidence_level
)
print("=" * 80)

In [ ]:
cate_models_polynomial["Linear Specification (Interactions)"].summary()

In [ ]:
cate_models_polynomial["PLR Specification"].blp_model.summary()

In [ ]:
latex_args = {
    "index": True,
    "float_format": "%.3f",
    "column_format": "lcccccc",
}

print(df_specification_1.to_latex(**latex_args))
print(df_specification_2.to_latex(**latex_args))
print(df_specification_3.to_latex(**latex_args))

In [ ]:
summary_list = []
for model_name, model in cate_models_polynomial.items():
    if "Linear" in model_name:
        tmp_summary = summarize_lin_model(model, level=confidence_level)
    else:
        tmp_summary = summarize_lin_model(model.blp_model, level=confidence_level)
    for index, row in tmp_summary.iterrows():
        variable = index
        summary_list.append(
            {
                "model": model_name,
                "variable": variable,
                "coef": row["coef"],
                "Std.Err.": row["std err"],
                "lower": row[ci_names[0]],  # Adjusted to match the column name
                "upper": row[ci_names[1]],  # Adjusted to match the column name
            }
        )

cate_summary_df = pd.DataFrame(summary_list)

In [ ]:
cate_summary_df

In [ ]:
models = cate_summary_df["model"].unique()
variables = cate_summary_df["variable"].unique()
plt.figure(figsize=(12, 6))
bar_width = 0.15
x_positions = np.arange(len(variables))
for i, model in enumerate(models):
    df_model = cate_summary_df[cate_summary_df["model"] == model]
    if not df_model.empty:
        plt.errorbar(
            x_positions + i * bar_width,
            df_model["coef"],
            yerr=[
                df_model["coef"] - df_model["lower"],
                df_model["upper"] - df_model["coef"],
            ],
            fmt="o",
            label=model,
            capsize=5,
            capthick=2,
            ecolor=palette[i % len(palette)],
            color=palette[i % len(palette)],
            zorder=2,
        )
plt.xticks(x_positions + (len(models) - 1) * bar_width / 2, variables, rotation=45)
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.title(f"Estimates and {ci_level_name} Confidence Intervals", fontsize=16)
plt.xlabel("Variable")
plt.ylabel("Coefficient")
plt.legend(title="Method")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"cluster_and_level_cate_estimates.png"))
plt.show()

### R² — Variance Explained by CATE Drivers

This cell decomposes how much of the CATE variation is explained
by different variable groups:

- **Embeddings** — how much of individual elasticity is driven
  by the multimodal product representation
- **Similarities** — how much is driven by cluster membership
- **Lagged variables** — how much is driven by past demand/price history

**Our findings:**
- Linear Spec: embeddings explain ~80%, similarities ~67%, lags ~72%
- PLR Spec: embeddings ~61%, similarities ~31%, lags ~7%

The PLR specification suggests past demand/price history (lags) matters
less for CATE than the static product embedding — meaning product
identity (what kind of shoe it is) drives elasticity more than
its recent performance history.

In [ ]:
alpha_x_lin = cate_models_polynomial["Linear Specification"].predict(df_basis)
alpha_x_lin_interactions = cate_models_polynomial[
    "Linear Specification (Interactions)"
].predict(df_basis)
alpha_x_plr = cate_models_polynomial["PLR Specification"].blp_model.predict(df_basis)

print("Linear Specification")
model_lin_emb = sm.OLS(alpha_x_lin, sm.add_constant(df_dynamic[embeddings])).fit()
model_lin_sim = sm.OLS(
    alpha_x_lin, sm.add_constant(df_dynamic[controls_similarities])
).fit()

print("Variance explained by embeddings: ", model_lin_emb.rsquared)
print("Variance explained by similarities: ", model_lin_sim.rsquared)

print("\nLinear Specification with interactions")
model_lin_interactions_emb = sm.OLS(
    alpha_x_lin_interactions, sm.add_constant(df_dynamic[embeddings])
).fit()
model_lin_interactions_sim = sm.OLS(
    alpha_x_lin_interactions, sm.add_constant(df_dynamic[controls_similarities])
).fit()
print("Variance explained by embeddings: ", model_lin_interactions_emb.rsquared)
print("Variance explained by similarities: ", model_lin_interactions_sim.rsquared)

print("\nPLR Specification")
model_plr_emb = sm.OLS(alpha_x_plr, sm.add_constant(df_dynamic[embeddings])).fit()
model_plr_sim = sm.OLS(
    alpha_x_plr, sm.add_constant(df_dynamic[controls_similarities])
).fit()
print("Variance explained by embeddings: ", model_plr_emb.rsquared)
print("Variance explained by similarities: ", model_plr_sim.rsquared)


## Chi-Squared Test — Is Elasticity Heterogeneity Significant?

Two chi-squared tests check whether CATE variation is statistically real:

**Test 1 — Cluster similarities only (IDs 3-7):**
Tests whether cluster membership alone predicts significant heterogeneity.
- Linear Spec: p = 0.847  → NOT significant
- PLR Spec:    p = 0.184  → NOT significant

**Test 2 — All basis variables (IDs 1-7):**
Tests whether the full basis (lags + clusters) predicts heterogeneity.
- Linear Spec: p = 1.02e-06 → SIGNIFICANT
- PLR Spec:    p = 2.15e-05 → SIGNIFICANT

**Interpretation:** Cluster membership alone does not explain elasticity
heterogeneity, but when combined with lagged variables (past price and demand),
the heterogeneity is statistically real and significant.

In [ ]:
# select index with cluster similarity
cluster_treatment_ids = [
    i for i, variable in enumerate(df_basis.columns) if "similarity_cluster" in variable
]
cluster_treatment_ids

In [ ]:
lin_p_val = chi2_test(
    cate_models_polynomial["Linear Specification"], cluster_treatment_ids
)
lin_interactions_p_val = chi2_test(
    cate_models_polynomial["Linear Specification (Interactions)"], cluster_treatment_ids
)
tab_dml_p_val = chi2_test(
    cate_models_polynomial["PLR Specification"], cluster_treatment_ids
)

print(f"Linear Specification p-value: {lin_p_val} (sample size: {len(df_dynamic)})")
print(
    f"Linear Specification (Interactions) p-value: {lin_interactions_p_val} (sample size: {len(df_dynamic)})"
)
print(f"PLR Specification p-value: {tab_dml_p_val} (sample size: {len(df_dynamic)})")

In [ ]:
# REVISED: het_ids adjusted — basis no longer contains lag vars
# Original had 8 columns (intercept + 5 similarities + 2 lags) → ids [1..7]
# Revised has 6 columns (intercept + 5 similarities) → ids [1..5]

het_ids = [1, 2, 3, 4, 5]   # REVISED: was [1,2,3,4,5,6,7]

lin_p_val = chi2_test(cate_models_polynomial["Linear Specification"], het_ids)
lin_interactions_p_val = chi2_test(
    cate_models_polynomial["Linear Specification (Interactions)"], het_ids
)
tab_dml_p_val = chi2_test(cate_models_polynomial["PLR Specification"], het_ids)

print(f"Linear Specification p-value: {lin_p_val} (sample size: {len(df_dynamic)})")
print(f"Linear Specification (Interactions) p-value: {lin_interactions_p_val} (sample size: {len(df_dynamic)})")
print(f"PLR Specification p-value: {tab_dml_p_val} (sample size: {len(df_dynamic)})")


## Sorted Treatment Effects (GATES Quantile Plot)

This plot sorts all products by their estimated individual elasticity
and shows how elasticity varies across the product distribution.

**Reading the plot:**
- Left side (low quantile) → products with the most negative elasticity
  (most price-sensitive customers)
- Right side (high quantile) → products with least elastic or even
  positive elasticity
- The width of the CI band shows estimation uncertainty

**Our finding:** The most elastic products (bottom 10%) have elasticity
around -0.37, while the least elastic (top 10%) are near 0 or slightly
positive. This range of variation supports heterogeneous pricing recommendations.

In [ ]:
treatment_ids = [0, 1, 2] + cluster_treatment_ids
df_basis_sorted = df_basis.iloc[:, treatment_ids]  # .join(df_dynamic[["ASIN", "date"]])

In [ ]:
df_basis_sorted

In [ ]:
uniform_ci = False

plt.rcParams["figure.figsize"] = (
    15.0,
    7.5,
)  # Adjusted the figure size for side-by-side plots
# Determine the number of models to plot
num_plots = 1
# Create subplots with cate_vars as rows and models as columns
fig, axs = plt.subplots(nrows=num_plots, ncols=3, figsize=(12, 6 * num_plots))
# If there's only one model, axs won't be a list, so wrap it in a list
if num_plots == 1:
    axs = np.array([axs, axs])  # Create a 2-column array


# Linear model
df_lin_ci_pointwise = predict_cate(
    cate_models_polynomial["Linear Specification"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
)
df_lin_ci_pointwise_sorted = df_lin_ci_pointwise.apply(
    lambda x: x.sort_values().values, axis=0
)
df_lin_ci_uniform = predict_cate(
    cate_models_polynomial["Linear Specification"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
    joint=True,
)
df_lin_ci_uniform_sorted = df_lin_ci_uniform.apply(
    lambda x: x.sort_values().values, axis=0
)
lm_intercept = cate_models_polynomial["Linear Specification"].params["intercept"]
lm_intercept_ci = (
    cate_models_polynomial["Linear Specification"].conf_int().loc["intercept"]
)

# Tabular DML model
df_tab_dml_ci_pointwise = predict_cate(
    cate_models_polynomial["Linear Specification (Interactions)"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
)
df_tab_dml_ci_pointwise_sorted = df_tab_dml_ci_pointwise.apply(
    lambda x: x.sort_values().values, axis=0
)
df_tab_dml_ci_joint = predict_cate(
    cate_models_polynomial["Linear Specification (Interactions)"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
    joint=True,
)
df_tab_dml_ci_joint_sorted = df_tab_dml_ci_joint.apply(
    lambda x: x.sort_values().values, axis=0
)

tab_dml_intercept = cate_models_polynomial["Linear Specification"].params["intercept"]
tab_dml_intercept_ci = (
    cate_models_polynomial["Linear Specification"].conf_int().loc["intercept"]
)

# PLR Model
df_deep_dml_ci_pointwise = predict_cate(
    cate_models_polynomial["PLR Specification"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
)
df_deep_dml_ci_pointwise_sorted = df_deep_dml_ci_pointwise.apply(
    lambda x: x.sort_values().values, axis=0
)
df_deep_dml_ci_joint = predict_cate(
    cate_models_polynomial["PLR Specification"],
    df_basis_sorted,
    treatment_ids=treatment_ids,
    level=confidence_level,
    joint=True,
)
df_deep_dml_ci_joint_sorted = df_deep_dml_ci_joint.apply(
    lambda x: x.sort_values().values, axis=0
)
deep_dml_intercept = cate_models_polynomial["PLR Specification"].blp_model.params[
    "intercept"
]
deep_dml_intercept_ci = (
    cate_models_polynomial["PLR Specification"].blp_model.conf_int().loc["intercept"]
)

# Calculate the y-axis limits for this row
y_min = min(
    df_lin_ci_pointwise[ci_names[0]].min(),
    df_lin_ci_uniform[ci_names[0]].min(),
    df_tab_dml_ci_pointwise[ci_names[0]].min(),
    df_tab_dml_ci_joint[ci_names[0]].min(),
    df_deep_dml_ci_pointwise[ci_names[0]].min(),
    df_deep_dml_ci_joint[ci_names[0]].min(),
)
y_max = max(
    df_lin_ci_pointwise[ci_names[1]].max(),
    df_lin_ci_uniform[ci_names[1]].max(),
    df_tab_dml_ci_pointwise[ci_names[1]].max(),
    df_tab_dml_ci_joint[ci_names[1]].max(),
    df_deep_dml_ci_pointwise[ci_names[1]].max(),
    df_deep_dml_ci_joint[ci_names[1]].max(),
)

# further adjust y-axis limits
y_min = max(y_min, -2)
y_max = min(y_max, 0.5)
percentiles_idx = np.linspace(0, 1, df_dynamic.shape[0])
percentiles_idx_full = np.linspace(0, 1, df_dynamic.shape[0])
# Plot on the first column
i = 0
ax0 = axs[i, 0]
ax0.plot(
    percentiles_idx_full, df_lin_ci_pointwise_sorted["effect"], label="Estimated Effect"
)
ax0.fill_between(
    percentiles_idx_full,
    df_lin_ci_pointwise_sorted[ci_names[0]],
    df_lin_ci_pointwise_sorted[ci_names[1]],
    color="b",
    alpha=0.3,
    label="Confidence Interval",
)
if uniform_ci:
    ax0.fill_between(
        percentiles_idx_full,
        df_lin_ci_uniform_sorted[ci_names[0]],
        df_lin_ci_uniform_sorted[ci_names[1]],
        color="b",
        alpha=0.1,
        label="Uniform Confidence Interval",
    )
ax0.axhline(y=lm_intercept, color="r", linestyle="--", label="ATE")
ax0.axhline(y=lm_intercept_ci[0], color="r", linestyle="dotted")
ax0.axhline(y=lm_intercept_ci[1], color="r", linestyle="dotted")
ax0.set_title("Linear Specification")
ax0.set_xlabel("Percentile index")
ax0.set_ylabel(f"Effect and {ci_level_name}-CI")
ax0.set_ylim(y_min, y_max)
ax1 = axs[i, 1]
ax1.plot(
    percentiles_idx_full,
    df_tab_dml_ci_pointwise_sorted["effect"],
    label="Estimated Effect",
)
ax1.fill_between(
    percentiles_idx_full,
    df_tab_dml_ci_pointwise_sorted[ci_names[0]],
    df_tab_dml_ci_pointwise_sorted[ci_names[1]],
    color="b",
    alpha=0.3,
    label="Confidence Interval",
)
if uniform_ci:
    ax1.fill_between(
        percentiles_idx_full,
        df_tab_dml_ci_joint_sorted[ci_names[0]],
        df_tab_dml_ci_joint_sorted[ci_names[1]],
        color="b",
        alpha=0.1,
        label="Uniform Confidence Interval",
    )
ax1.axhline(y=tab_dml_intercept, color="r", linestyle="--", label="Average Effect")
ax1.axhline(y=tab_dml_intercept_ci[0], color="r", linestyle="dotted")
ax1.axhline(y=tab_dml_intercept_ci[1], color="r", linestyle="dotted")
ax1.set_title("Linear Specification (Interactions)")
ax1.set_xlabel("Percentile index")
ax1.set_ylabel(f"Effect and {ci_level_name}-CI")
ax1.set_ylim(y_min, y_max)

ax2 = axs[i, 2]
ax2.plot(
    percentiles_idx_full,
    df_deep_dml_ci_pointwise_sorted["effect"],
    label="Estimated Effect",
)
ax2.fill_between(
    percentiles_idx_full,
    df_deep_dml_ci_pointwise_sorted[ci_names[0]],
    df_deep_dml_ci_pointwise_sorted[ci_names[1]],
    color="b",
    alpha=0.3,
    label="Confidence Interval",
)
if uniform_ci:
    ax2.fill_between(
        percentiles_idx_full,
        df_deep_dml_ci_joint_sorted[ci_names[0]],
        df_deep_dml_ci_joint_sorted[ci_names[1]],
        color="b",
        alpha=0.1,
        label="Uniform Confidence Interval",
    )
ax2.axhline(y=deep_dml_intercept, color="r", linestyle="--", label="Average Effect")
ax2.axhline(y=deep_dml_intercept_ci[0], color="r", linestyle="dotted")
ax2.axhline(y=deep_dml_intercept_ci[1], color="r", linestyle="dotted")

ax2.set_title("PLR Specification (Boosted Trees)")
ax2.set_xlabel("Percentile index")
ax2.set_ylabel(f"Effect and {ci_level_name}-CI")
ax2.set_ylim(y_min, y_max)

# Create a single legend for all plots and place it at the lower center
handles, labels = axs[
    -1, 1
].get_legend_handles_labels()  # Collect handles and labels from the last subplot
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=4)
# Adjust layout to make space for the legend
plt.tight_layout(rect=[0, 0.0, 1, 1])  # Leave space at the bottom for the legend

# Save the figure with tight bounding box to include the legend
plt.savefig(os.path.join(output_dir, "cluster_sorted_effects.png"), bbox_inches="tight")
plt.show()

In [ ]:
quantiles = np.arange(start=0.01, stop=1, step=0.01)
m = len(quantiles)
pes = df_tab_dml_ci_pointwise["effect"]
spes = df_tab_dml_ci_pointwise["effect"].quantile(quantiles).values
spes

---

# Sensitivity Analysis

Sensitivity analysis tests whether our ATE estimate is robust to
**unobserved confounding** — hidden variables that affect both price and demand.

**Why this matters:** Even with DoubleML, if there are unobserved product
quality signals (e.g. seller reputation, listing quality) that correlate
with both price and sales rank, our estimate could be biased.

**Parameters:**
- `cf_y` — fraction of outcome variance explained by unobserved confounder
- `cf_d` — fraction of treatment variance explained by unobserved confounder
- `rho` — correlation between confounders in Q and P equations

**Three scenarios tested:**
- Mild (0.01, 0.01) — small confounding
- Medium (0.025, 0.025) — moderate confounding
- Strong (0.05, 0.05) — strong confounding

**Our result:** Under mild and medium confounding with rho=1.0, our estimate
of -0.067 remains negative. Only under strong confounding does the CI
touch zero — suggesting reasonable robustness to omitted variable bias.

In [ ]:
plr_obj.sensitivity_analysis(
    cf_y=0.03,
    cf_d=0.03,
    rho=1.0,
    level=0.9,
    null_hypothesis=0.0
)
print(plr_obj.sensitivity_summary)

In [ ]:
sigma_l = tab_dml_obj_embeddings_add_controls.sensitivity_elements["sigma2"]
sigma_s = tab_dml_obj_embeddings.sensitivity_elements["sigma2"]

nu_l = tab_dml_obj_embeddings_add_controls.sensitivity_elements["nu2"]
nu_s = tab_dml_obj_embeddings.sensitivity_elements["nu2"]

delta_theta = tab_dml_obj_embeddings_add_controls.coef - tab_dml_obj_embeddings.coef

rho_hat = np.abs(delta_theta) / (np.sqrt(sigma_s - sigma_l) * np.sqrt(nu_l - nu_s)).squeeze()

print(f"Implied rho: {rho_hat}")

In [ ]:
r2_scores_l = tab_dml_obj_embeddings_add_controls.evaluate_learners(metric=r2)
print(
    f"R2 Outcome: {r2_scores_l['ml_l'][0, 0]} R2 Treatment: {r2_scores_l['ml_m'][0, 0]}\n"
)

r2_y_l = r2_scores_l["ml_l"][0, 0]
r2_d_l = r2_scores_l["ml_m"][0, 0]


r2_scores_s = tab_dml_obj_embeddings.evaluate_learners(metric=r2)
print(
    f"R2 Outcome: {r2_scores_s['ml_l'][0, 0]} R2 Treatment: {r2_scores_s['ml_m'][0, 0]}\n"
)

r2_y_s = r2_scores_s["ml_l"][0, 0]
r2_d_s = r2_scores_s["ml_m"][0, 0]

In [ ]:
partial_r2_y = (r2_y_l - r2_y_s) / (1 - r2_y_s)
partial_r2_d = (r2_d_l - r2_d_s) / (1 - r2_d_s)

print(f"Partial R2 Outcome: {partial_r2_y}")
print(f"Partial R2 Treatment: {partial_r2_d}")

In [ ]:
# Implied RV

implied_RV = np.sqrt(partial_r2_y * (partial_r2_d / (1 - partial_r2_d)))
print(f"Implied RV: {implied_RV}")

In [ ]:
# Scaled implied RV

scaled_implied_RV = implied_RV * rho_hat
print(f"Scaled Implied RV: {scaled_implied_RV}")

In [ ]:
sensitivity_res = dict()

rho_vec = [1.0, 0.5, 0.2]
scenarios = [(0.01, 0.01), (0.025, 0.025), (0.05, 0.05)]

theta_lower = []
theta_upper = []
ci_lower = []
ci_upper = []
R2_y = []
R2_d = []
rho_list = []

for rho in rho_vec:
    for cf_y, cf_d in scenarios:
        plr_obj.sensitivity_analysis(
            cf_y=cf_y,
            cf_d=cf_d,
            rho=rho,
            level=confidence_level,
            null_hypothesis=0.0
        )
        theta_lower.append(plr_obj.sensitivity_params["theta"]["lower"][0])
        theta_upper.append(plr_obj.sensitivity_params["theta"]["upper"][0])
        ci_lower.append(plr_obj.sensitivity_params["ci"]["lower"][0])
        ci_upper.append(plr_obj.sensitivity_params["ci"]["upper"][0])
        R2_y.append(cf_y)
        R2_d.append(cf_d)
        rho_list.append(rho)


In [ ]:
df_sensitivity = pd.DataFrame({
    "theta": plr_obj.coef[0],
    "theta_lower": theta_lower,
    "theta_upper": theta_upper,
    "ci_lower": ci_lower,
    "ci_upper": ci_upper,
    "R2_y": R2_y,
    "R2_d": R2_d,
    "rho": rho_list,
})
df_sensitivity.head()

In [ ]:
scenario_names = ["mild", "medium", "strong"]

latex_args = {
    "index": False,
    "float_format": "%.3f",
    "column_format": "lcccccc",
    "escape": False,  # Allow LaTeX in column names
}

ci_lower_label = f"$[ {{{100*(1-confidence_level):.1f}}}\\% $"
ci_upper_label = f"${{{100*(confidence_level):.1f}}}\\% ]$"

for rho_val in rho_vec:
    df_rho = df_sensitivity[df_sensitivity["rho"] == rho_val].copy()
    # Add scenario column with names
    df_rho["Scenario ($R^2_Y$, $R^2_D$)"] = [
        f"{name} $({str(r2y).lstrip('0')}, {str(r2d).lstrip('0')})$"
        for name, r2y, r2d in zip(scenario_names, df_rho["R2_y"], df_rho["R2_d"])
    ]
    # Rename columns for LaTeX
    df_rho = df_rho.rename(
        columns={
            "theta": r"$\theta$",
            "theta_lower": r"$\theta_-$",
            "theta_upper": r"$\theta_+$",
            "ci_lower": ci_lower_label,
            "ci_upper": ci_upper_label,
        }
    )
    # Reorder columns as requested
    df_rho = df_rho[
        [
            "Scenario ($R^2_Y$, $R^2_D$)",
            ci_lower_label,
            r"$\theta_-$",
            r"$\theta$",
            r"$\theta_+$",
            ci_upper_label,
        ]
    ]
    caption = (
        f"Sensitivity analysis for $\\rho={rho_val}$: "
    )
    latex_str = df_rho.to_latex(
        caption=caption,
        label=f"tab:sensitivity_rho_{str(rho_val).replace('.', '')}",
        **latex_args,
    )
    # Insert \hline\hline after \begin{tabular} and replace \toprule/\bottomrule
    latex_str = latex_str.replace("\\toprule", "\\hline\\hline")
    latex_str = latex_str.replace("\\midrule", "\\midrule")
    latex_str = latex_str.replace("\\bottomrule", "\\hline\\hline")
    latex_str = latex_str.replace("\\begin{table}", "\\begin{table}[h]\n\\centering")
    print(latex_str)

---

## ★ Additional Step A — Price Change vs Demand Change Scatter Plot

**Purpose:** Visualize the relationship between Delta_P_bb_t (price change) and
Delta_Q_t (demand change) — matching the revised Delta specification.

The slope line uses the best DoubleML estimate anchored at the mean of
Delta_P and Delta_Q in the val set.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── REVISION: Use Delta variables to match revised specification ───────────
# ORIGINAL: p_vals = df_dynamic["P_bb_t"], q_vals = df_dynamic["Q_t"]
# REVISED:  p_vals = df_dynamic["Delta_P_bb_t"], q_vals = df_dynamic["Delta_Q_t"]
# REASON:   Scatter must reflect the actual outcome/treatment used in DoubleML

p_vals = df_dynamic["Delta_P_bb_t"].values   # REVISED
q_vals = df_dynamic["Delta_Q_t"].values       # REVISED

# DoubleML elasticity estimate
elasticity = tab_dml_obj_embeddings_add_controls.coef[0]
ci_lo = ci_tab_dml_embeddings_add_controls["lower"].values[0]
ci_hi = ci_tab_dml_embeddings_add_controls["upper"].values[0]

# Slope line anchored at means
p_mean = np.mean(p_vals)
q_mean = np.mean(q_vals)
p_range = np.linspace(np.percentile(p_vals, 2), np.percentile(p_vals, 98), 100)
q_line  = q_mean + elasticity * (p_range - p_mean)
q_lo    = q_mean + ci_lo * (p_range - p_mean)
q_hi    = q_mean + ci_hi * (p_range - p_mean)

fig, ax = plt.subplots(figsize=(10, 6))

# Scatter subsample for clarity
np.random.seed(42)
idx = np.random.choice(len(p_vals), size=min(2000, len(p_vals)), replace=False)
ax.scatter(p_vals[idx], q_vals[idx], alpha=0.15, s=8,
           color=palette[0], label="Product observations (val set)")

ax.plot(p_range, q_line, color=palette[1], linewidth=2.5,
        label=f"DoubleML elasticity slope: {elasticity:.3f}")
ax.fill_between(p_range, q_lo, q_hi, alpha=0.15, color=palette[1],
                label=f"90% CI [{ci_lo:.3f}, {ci_hi:.3f}]")
ax.axvline(p_mean, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(q_mean, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Delta Log Price — Delta_P_bb_t", fontsize=12)
ax.set_ylabel("Delta Log Sales Rank — Delta_Q_t", fontsize=12)
ax.set_title("Price Change vs Demand Change — DoubleML Elasticity Visualization", fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{output_dir}/scatter_DeltaP_vs_DeltaQ_elasticity.png", dpi=150)
plt.show()
print(f"Elasticity estimate: {elasticity:.4f}")
print(f"Interpretation: 1% change in price -> {elasticity:.4f}% change in sales rank")


---

## ★ Additional Step B — Forest Plot of All DoubleML Specifications

**Purpose:** Present all DoubleML coefficient estimates side by side with confidence intervals
in a clean horizontal forest plot — easier to read than the original vertical errorbar chart.

A forest plot is the standard way to summarize multiple model estimates in econometrics papers.
It immediately shows which models agree, which are more uncertain, and whether zero is excluded.
Color coding: blue = statistically significant at 90%, orange = not significant.

In [ ]:
all_dml = pd.concat([
    ci_tab_dml_embeddings.assign(model="PLR + Embeddings"),
    ci_tab_dml_add_controls.assign(model="PLR + Controls"),
    ci_tab_dml_embeddings_add_controls.assign(model="PLR + Emb + Controls"),
    ci_tab_dml_sim_add_controls.assign(model="PLR + Sim + Controls"),
    ci_tab_dml_pca_add_controls.assign(model="PLR + PCA + Controls"),
], axis=0).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))

for i, (_, row) in enumerate(all_dml.iterrows()):
    is_sig = row["lower"] > 0 or row["upper"] < 0
    c = palette[1] if is_sig else palette[0]
    ax.plot([row["lower"], row["upper"]], [i, i], color=c, linewidth=3, solid_capstyle="round")
    ax.plot(row["coef"], i, "o", color=c, markersize=9, zorder=5)

ax.axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.6)
ax.set_yticks(range(len(all_dml)))
ax.set_yticklabels(all_dml["model"], fontsize=11)
ax.set_xlabel("Price Elasticity Estimate", fontsize=12)
ax.set_title("DoubleML Elasticity Estimates — Forest Plot (90% CI)", fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{output_dir}/forest_plot_dml_elasticity.png", dpi=150)
plt.show()
print("Saved forest_plot_dml_elasticity.png")

---

## ★ Additional Step C — Elasticity by Cluster (GATES Visualization)

**Purpose:** Show how price elasticity differs across the 5 product clusters.

GATES (Group Average Treatment Effects) estimates a separate elasticity for each cluster.
This answers: *"Are some shoe types more price-sensitive than others?"*

A cluster with a more negative elasticity has more elastic demand — customers respond
more strongly to price changes. This is directly useful for pricing strategy recommendations.

In [ ]:
cluster_names = [f"Cluster {i}" for i in range(5)]

coefs  = gates_0_summ["coef"].values
lowers = gates_0_summ["err_lower"].values
uppers = gates_0_summ["err_upper"].values

fig, ax = plt.subplots(figsize=(10, 5))

for i in range(len(coefs)):
    is_sig = lowers[i] > 0 or uppers[i] < 0
    c = palette[1] if is_sig else palette[0]
    ax.plot([lowers[i], uppers[i]], [i, i], color=c, linewidth=3, solid_capstyle="round")
    ax.plot(coefs[i], i, "o", color=c, markersize=9, zorder=5)

ax.axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.6)

avg_e = tab_dml_obj_embeddings_add_controls.coef[0]
ax.axvline(avg_e, color="red", linewidth=1.5, linestyle=":",
           label=f"Average elasticity: {avg_e:.3f}")

ax.set_yticks(range(len(cluster_names)))
ax.set_yticklabels(cluster_names, fontsize=11)
ax.set_xlabel("Price Elasticity (GATES)", fontsize=12)
ax.set_title("Heterogeneous Price Elasticity by Product Cluster", fontsize=13)
ax.legend(fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{output_dir}/gates_elasticity_by_cluster.png", dpi=150)
plt.show()
print("Saved gates_elasticity_by_cluster.png")

---

## ★ Additional Step D — CATE Distribution (Individual Elasticity Histogram)

**Purpose:** Show the full distribution of individual-level elasticities across all products.

While the average elasticity (ATE) is one number, individual products may have very
different elasticities. This histogram shows the spread — are most products similarly
elastic, or is there wide variation?

A wide spread supports heterogeneous pricing strategies. The sorted treatment effects
plot (right panel) shows what fraction of products have elastic vs inelastic demand.

In [ ]:
alpha_x_plr = cate_models_polynomial["PLR Specification"].blp_model.predict(df_basis)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(alpha_x_plr, bins=40, color=palette[0], edgecolor="white", alpha=0.8, density=True)
axes[0].axvline(np.mean(alpha_x_plr), color=palette[1], linewidth=2,
                linestyle="--", label=f"Mean: {np.mean(alpha_x_plr):.3f}")
axes[0].axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[0].set_xlabel("Individual Price Elasticity (CATE)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("Distribution of Individual Elasticities", fontsize=12)
axes[0].legend(fontsize=10)

sorted_cates = np.sort(alpha_x_plr)
axes[1].plot(np.linspace(0, 100, len(sorted_cates)), sorted_cates,
             color=palette[0], linewidth=1.5)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[1].axhline(np.mean(alpha_x_plr), color=palette[1], linewidth=1.5,
                linestyle="--", label=f"Mean: {np.mean(alpha_x_plr):.3f}")
axes[1].set_xlabel("Percentile of Products", fontsize=11)
axes[1].set_ylabel("Price Elasticity", fontsize=11)
axes[1].set_title("Sorted Treatment Effects", fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle("Heterogeneous Price Elasticities — CATE Distribution", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{output_dir}/cate_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean CATE:  {np.mean(alpha_x_plr):.4f}")
print(f"Std CATE:   {np.std(alpha_x_plr):.4f}")
print(f"Min CATE:   {np.min(alpha_x_plr):.4f}")
print(f"Max CATE:   {np.max(alpha_x_plr):.4f}")

---

## ★ Additional Step D2 — CATE Scatter Plot

**Purpose:** Visualize individual elasticities against price and sales rank, colored by cluster.

Two panels:
- Left: CATE vs log price — shows whether cheaper or more expensive shoes are more elastic
- Right: CATE vs log sales rank — shows whether popular or niche products are more elastic

Color coding by cluster reveals which product segments drive the heterogeneity found in Step D.
Products above zero on the Y axis have perverse elasticity (higher price → better rank),
which can happen for luxury or status goods.

In [ ]:
# ── Additional Step D2 — CATE Scatter Plot ────────────────────────────────
# REVISION: x-axis uses Delta_P_bb_t, y-axis uses Delta_Q_t to match spec

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_assignment = df_dynamic[[
    "similarity_cluster_0", "similarity_cluster_1",
    "similarity_cluster_2", "similarity_cluster_3",
    "similarity_cluster_4"
]].idxmax(axis=1).str.replace("similarity_cluster_", "").astype(int).values

p_vals_cate = df_dynamic["Delta_P_bb_t"].values   # REVISED
q_vals_cate = df_dynamic["Delta_Q_t"].values       # REVISED

# ── Left: CATE vs Delta Log Price ─────────────────────────────────────────
scatter = axes[0].scatter(
    p_vals_cate, alpha_x_plr,
    c=cluster_assignment, cmap="viridis",
    alpha=0.4, s=10
)
axes[0].axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[0].axhline(np.mean(alpha_x_plr), color="red", linewidth=1.5,
                linestyle="--", label=f"Mean CATE: {np.mean(alpha_x_plr):.3f}")
axes[0].set_xlabel("Delta Log Price — Delta_P_bb_t", fontsize=11)
axes[0].set_ylabel("Individual Elasticity (CATE)", fontsize=11)
axes[0].set_title("CATE vs Delta Log Price", fontsize=12)
axes[0].legend(fontsize=10)
plt.colorbar(scatter, ax=axes[0], label="Cluster")

# ── Right: CATE vs Delta Log Sales Rank ───────────────────────────────────
scatter2 = axes[1].scatter(
    q_vals_cate, alpha_x_plr,
    c=cluster_assignment, cmap="viridis",
    alpha=0.4, s=10
)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[1].axhline(np.mean(alpha_x_plr), color="red", linewidth=1.5,
                linestyle="--", label=f"Mean CATE: {np.mean(alpha_x_plr):.3f}")
axes[1].set_xlabel("Delta Log Sales Rank — Delta_Q_t", fontsize=11)
axes[1].set_ylabel("Individual Elasticity (CATE)", fontsize=11)
axes[1].set_title("CATE vs Delta Log Sales Rank", fontsize=12)
axes[1].legend(fontsize=10)
plt.colorbar(scatter2, ax=axes[1], label="Cluster")

plt.suptitle("Individual Price Elasticity (CATE) — Scatter Plots", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{output_dir}/cate_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved cate_scatter.png")


---

## ★ Additional Step D3 — CATE Scatter per Cluster

**Purpose:** Show individual elasticities vs price separately for each of the 5 clusters.

Each panel shows one cluster — allowing direct comparison of:
- Whether the price range differs across clusters (x-axis spread)
- Whether elasticity differs across clusters (y-axis level)
- The dashed line shows each cluster's mean CATE vs the global mean (red dotted)

This is the most detailed heterogeneity visualization in the notebook.

In [ ]:
# ── Additional Step D3 — CATE Scatter per Cluster ─────────────────────────
# REVISION: x-axis uses Delta_P_bb_t to match revised specification

cluster_assignment = df_dynamic[[
    "similarity_cluster_0", "similarity_cluster_1",
    "similarity_cluster_2", "similarity_cluster_3",
    "similarity_cluster_4"
]].idxmax(axis=1).str.replace("similarity_cluster_", "").astype(int).values

p_vals_cate = df_dynamic["Delta_P_bb_t"].values   # REVISED
q_vals_cate = df_dynamic["Delta_Q_t"].values       # REVISED

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)
colors = plt.cm.viridis([0.0, 0.25, 0.5, 0.75, 1.0])

global_mean = np.mean(alpha_x_plr)

for c_id in range(5):
    ax = axes[c_id]
    mask = cluster_assignment == c_id

    p_c    = p_vals_cate[mask]
    cate_c = alpha_x_plr[mask]

    ax.scatter(p_c, cate_c, color=colors[c_id], alpha=0.4, s=12)

    c_mean = np.mean(cate_c)
    ax.axhline(c_mean, color=colors[c_id], linewidth=2,
               linestyle="--", label=f"Cluster mean: {c_mean:.3f}")
    ax.axhline(global_mean, color="red", linewidth=1.2,
               linestyle=":", alpha=0.7, label=f"Global mean: {global_mean:.3f}")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="-", alpha=0.3)

    ax.set_title(f"Cluster {c_id}\n(n={mask.sum()})", fontsize=11)
    ax.set_xlabel("Delta Log Price (Delta_P_bb_t)", fontsize=10)
    if c_id == 0:
        ax.set_ylabel("CATE (individual elasticity)", fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("CATE vs Delta Log Price — Separated by Cluster\n(red dotted = global mean, dashed = cluster mean)",
             fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(f"{output_dir}/cate_scatter_per_cluster.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved cate_scatter_per_cluster.png")

print("\nCluster-level CATE summary:")
print("-" * 45)
for c_id in range(5):
    mask = cluster_assignment == c_id
    cate_c = alpha_x_plr[mask]
    p_c = p_vals_cate[mask]
    print(f"  Cluster {c_id}: n={mask.sum():4d} | "
          f"mean CATE={np.mean(cate_c):7.4f} | "
          f"mean Delta_P={np.mean(p_c):.3f} | "
          f"std CATE={np.std(cate_c):.4f}")


---

## ★ Additional Step E — Clean Summary Table for Presentation

**Purpose:** Print a single clean summary of all key results in one place.

This is useful for your presentation to Guy — all elasticity estimates, GATES,
and CATE statistics collected into a structured printout you can directly reference.

In [ ]:
print("=" * 65)
print("SUMMARY — Price Elasticity Results: Women Shoes Size 8")
print("=" * 65)

print("\n[1] Average DoubleML Elasticity Estimates (PLR Model):")
print("-" * 65)
for _, row in all_dml.iterrows():
    is_sig = row["lower"] > 0 or row["upper"] < 0
    sig = "significant" if is_sig else "not significant"
    print(f"  {row['model']:<35} {row['coef']:>7.4f}  [{row['lower']:.4f}, {row['upper']:.4f}]  {sig}")

print("\n[2] CATE Distribution (PLR Specification):")
print("-" * 65)
print(f"  Mean: {np.mean(alpha_x_plr):.4f}  |  Std: {np.std(alpha_x_plr):.4f}")
print(f"  Range: [{np.min(alpha_x_plr):.4f}, {np.max(alpha_x_plr):.4f}]")

print("\n[3] GATES — Elasticity by Cluster:")
print("-" * 65)
for i, (_, row) in enumerate(gates_0_summ.iterrows()):
    is_sig = row["err_lower"] > 0 or row["err_upper"] < 0
    sig = "sig" if is_sig else "n.s."
    print(f"  Cluster {i}: {row['coef']:>7.4f}  [{row['err_lower']:.4f}, {row['err_upper']:.4f}]  {sig}")

print("\n[4] Interpretation:")
print("-" * 65)
best_e = tab_dml_obj_embeddings_add_controls.coef[0]
print(f"  Best estimate: {best_e:.4f}")
print(f"  A 1% price increase -> {best_e:.4f}% change in log sales rank")
print(f"  (negative = fewer units sold when price rises)")
print("=" * 65)

---

## ★ Additional Step F — Why Do Clusters Differ?

**Key finding from our CATE analysis:**
Cluster 4 shows the most elastic demand (mean CATE = -0.163) while
Cluster 2 shows the least elastic (mean CATE = -0.135), despite similar
price ranges in some segments.

Since brand is already encoded in the text field (RoBERTa encodes it),
the difference must come from product type, fulfillment method, or visual characteristics.

We investigate across three dimensions:
- **Method 1** — Tabular features: rating, reviews, FBA status
- **Method 2** — Subcategory distribution: are they different shoe types?
- **Method 3** — PCA embedding space: do they separate in embedding dimensions?

### Method 1 — Tabular Feature Comparison

If Cluster 2 has a higher FBA ratio or more reviews, that could explain
why its customers are more price-sensitive — FBA products are more
discoverable and comparison-shopped more actively.

In [ ]:
TAB_COLS = [
    "RATING", "REVIEW_COUNT",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
    "Lightning Deals: Upcoming Deal",
    "Buy Box: Is FBA",
]

cluster_assignment = df_dynamic[[
    "similarity_cluster_0", "similarity_cluster_1",
    "similarity_cluster_2", "similarity_cluster_3",
    "similarity_cluster_4"
]].idxmax(axis=1).str.replace("similarity_cluster_", "").astype(int)

df_dynamic["cluster"] = cluster_assignment

tab_available = [c for c in TAB_COLS if c in df_dynamic.columns]
summary = df_dynamic.groupby("cluster")[tab_available].mean().round(3)
print("=== Mean tabular features by cluster ===")
print(summary.to_string())

print()
print("=== Cluster 2 vs Cluster 4 — direct comparison ===")
comparison = pd.DataFrame({
    "Cluster 2": summary.loc[2],
    "Cluster 4": summary.loc[4],
    "Diff (2-4)": (summary.loc[2] - summary.loc[4]).round(3),
    "Diff %":     ((summary.loc[2] - summary.loc[4]) / (summary.loc[4].abs() + 1e-9) * 100).round(1),
})
print(comparison.to_string())

In [ ]:
fig, axes = plt.subplots(1, len(tab_available), figsize=(18, 4))
for j, col in enumerate(tab_available):
    vals = [summary.loc[2, col], summary.loc[4, col]]
    max_val = max(abs(v) for v in vals) + 1e-9
    bars = axes[j].bar(["C2", "C4"], vals, color=["#c0504d","#9dc3e6"],
                        width=0.5, edgecolor="white")
    short = col.replace("Count of retrieved live offers: New, ","").replace(": Current","")
    axes[j].set_title(short, fontsize=9)
    for bar, v in zip(bars, vals):
        axes[j].text(bar.get_x()+bar.get_width()/2,
                     bar.get_height()+0.01*max_val,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)
plt.suptitle("Tabular Features: Cluster 2 (red) vs Cluster 4 (blue)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{output_dir}/cluster2_vs4_tabular.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved cluster2_vs4_tabular.png")

### Method 2 — Subcategory Distribution

The subcategory field (`subcat_aggregated`) tells us the shoe type.
If Clusters 2 and 4 contain different subcategories at the same price point,
the elasticity gap is genuine product-type heterogeneity — not measurement noise.

In [ ]:
print("=== Subcategory distribution: ALL clusters ===")
for c in range(5):
    mask = cluster_assignment == c
    subcat = df_dynamic[mask]["subcat_aggregated"].value_counts(normalize=True).round(3)
    mean_p = df_dynamic[mask]["P_bb_t"].mean()
    print(f"\nCluster {c} (n={mask.sum()}, ~${np.exp(mean_p):.2f}):")
    for s, v in subcat.items():
        bar = chr(9608) * int(v * 30)
        print(f"  {str(s):<35} {bar} {v:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cate_vals = {2: -0.2995, 4: -0.2337}
for idx, c in enumerate([2, 4]):
    mask = cluster_assignment == c
    subcat = df_dynamic[mask]["subcat_aggregated"].value_counts(normalize=True)
    color = "#c0504d" if c == 2 else "#9dc3e6"
    axes[idx].barh(subcat.index, subcat.values, color=color, edgecolor="white")
    axes[idx].set_title(f"Cluster {c}  (n={mask.sum()}, ~$21.59, CATE={cate_vals[c]:.4f})", fontsize=11)
    axes[idx].set_xlabel("Share", fontsize=10)
    axes[idx].set_xlim(0, 1)
    for i, v in enumerate(subcat.values):
        axes[idx].text(v+0.01, i, f"{v:.2f}", va="center", fontsize=9)
plt.suptitle("Subcategory Distribution: Cluster 2 vs Cluster 4\n(same price ~$21.59, different elasticity)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{output_dir}/cluster2_vs4_subcat.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved cluster2_vs4_subcat.png")

### Method 3 — PCA Embedding Space

If Clusters 2 and 4 separate clearly in one PCA dimension, that dimension
captures the latent product characteristic driving the elasticity gap.
Larger mean differences (shown in subplot titles) = stronger embedding separation.

In [ ]:
pca_cols = [c for c in ["pca_0","pca_1","pca_2","pca_3","pca_4"] if c in df_dynamic.columns]
fig, axes = plt.subplots(1, len(pca_cols), figsize=(16, 4), sharey=False)

for j, pca_col in enumerate(pca_cols):
    n2 = (cluster_assignment==2).sum()
    n4 = (cluster_assignment==4).sum()
    for c, color, label in [
        (2, "#c0504d", f"Cluster 2 (n={n2})"),
        (4, "#9dc3e6", f"Cluster 4 (n={n4})"),
    ]:
        vals = df_dynamic[cluster_assignment==c][pca_col].values
        axes[j].hist(vals, bins=30, alpha=0.6, color=color, label=label, density=True)
    m2 = df_dynamic[cluster_assignment==2][pca_col].mean()
    m4 = df_dynamic[cluster_assignment==4][pca_col].mean()
    axes[j].axvline(m2, color="#c0504d", linewidth=2, linestyle="--")
    axes[j].axvline(m4, color="#3266ad", linewidth=2, linestyle="--")
    axes[j].set_title(f"{pca_col}\nC2={m2:.3f} C4={m4:.3f}\ngap={abs(m2-m4):.3f}", fontsize=9)
    if j == 0:
        axes[j].legend(fontsize=8)

plt.suptitle("PCA Embedding Distributions: Cluster 2 (red) vs Cluster 4 (blue)\n"
             "(dashed = cluster mean, larger gap = stronger embedding separation)",
             fontsize=12, y=1.05)
plt.tight_layout()
plt.savefig(f"{output_dir}/cluster2_vs4_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved cluster2_vs4_pca.png")

### Summary — Cluster Elasticity Investigation

**What each result means:**

If **Method 1** shows different FBA ratios — Amazon-fulfilled
products are more elastic because customers comparison-shop more actively.

If **Method 2** shows different subcategories — the elasticity gap is
genuine product-type heterogeneity. Two shoe types at different price
points have different demand curves — an economically meaningful finding.

If **Method 3** shows clear PCA separation —
the multimodal embedding captured a latent product dimension that pure
price data cannot see. This validates the embedding approach.

---

**Key message for your presentation:**

> "Our multimodal embeddings identify product segments with meaningfully
> different price elasticities across the 5 clusters (-0.163 to -0.135).
> All ATE estimates are statistically significant, with the best specification
> (PLR + Embeddings + Controls) yielding an elasticity of -0.066.
> This demonstrates the value of the embedding-based approach over purely
> tabular methods — the fused text + image + tabular representation reveals
> economic heterogeneity that price alone cannot explain."


In [ ]:
# Check cluster distribution in train vs val
train_clusters = df_full_train[[
    "similarity_cluster_0","similarity_cluster_1",
    "similarity_cluster_2","similarity_cluster_3",
    "similarity_cluster_4"
]].idxmax(axis=1).value_counts(normalize=True).sort_index()

val_clusters = df_full_val[[
    "similarity_cluster_0","similarity_cluster_1",
    "similarity_cluster_2","similarity_cluster_3",
    "similarity_cluster_4"
]].idxmax(axis=1).value_counts(normalize=True).sort_index()

print("Cluster distribution — Train vs Val:")
comparison = pd.DataFrame({"Train": train_clusters, "Val": val_clusters})
print(comparison.round(3))